In [1]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO

# --- 1. Загрузка твоих дедуплицированных данных ---
fused_json_path = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train_fused_deduplicated.json"
train_img_dir = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\train_images"

coco = COCO(fused_json_path)
df_folds = pd.read_csv("clean_fused_folds.csv")

# Разбиваем на train/val (например, Fold 0 — это валидация)
val_img_ids = df_folds[df_folds['fold'] == 0]['image_id'].tolist()
train_img_ids = df_folds[df_folds['fold'] != 0]['image_id'].tolist()

# Достаем все аннотации для train
train_ann_ids = []
for img_id in train_img_ids:
    train_ann_ids.extend(coco.getAnnIds(imgIds=[img_id]))

print(f"Тренировочных филаментов (кропов): {len(train_ann_ids)}")

# --- 2. Класс Dataset для кропов ---
class GTCropDataset(Dataset):
    def __init__(self, coco_api, img_dir, ann_ids, transform, pad_ratio=0.20, crop_size=512):
        self.coco = coco_api
        self.img_dir = img_dir
        self.ann_ids = ann_ids
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.crop_size = crop_size
        self.img_cache = {} # Кэш для ускорения

    def __len__(self):
        return len(self.ann_ids)

    def __getitem__(self, idx):
        ann_id = self.ann_ids[idx]
        ann = self.coco.loadAnns([ann_id])[0]
        img_id = ann['image_id']
        img_info = self.coco.loadImgs([img_id])[0]
        fname = img_info['file_name']
        
        # Читаем и кэшируем изображение
        if fname not in self.img_cache:
            img_path = os.path.join(self.img_dir, fname)
            raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            # Твоя CLAHE обработка:
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            disk_mask = raw_img > 15
            proc_img = raw_img.copy()
            if np.any(disk_mask):
                proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
            
            proc_img = proc_img.astype(np.float32) / 255.0
            self.img_cache[fname] = np.stack([proc_img] * 3, axis=-1)
            
        img = self.img_cache[fname]
        h, w, _ = img.shape
        full_mask = self.coco.annToMask(ann)

        x, y, bw, bh = ann['bbox']
        
        # Вырезаем квадратный кроп с сохранением пропорций
        cx, cy = x + bw / 2, y + bh / 2
        max_dim = max(bw, bh) * (1 + self.pad_ratio)
        x1, y1 = max(0, int(cx - max_dim / 2)), max(0, int(cy - max_dim / 2))
        x2, y2 = min(w, int(cx + max_dim / 2)), min(h, int(cy + max_dim / 2))

        crop_img = img[y1:y2, x1:x2]
        crop_mask = full_mask[y1:y2, x1:x2]

        c = cv2.resize(crop_img, (self.crop_size, self.crop_size))
        mk = cv2.resize(crop_mask, (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST).astype('float32')

        out = self.transform(image=c, mask=mk)
        return out['image'], out['mask'].unsqueeze(0).float()

# Аугментации
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    ToTensorV2()
])

train_crops = GTCropDataset(coco, train_img_dir, train_ann_ids, train_tf)
train_loader = DataLoader(train_crops, batch_size=8, shuffle=True, num_workers=2)

c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\albumentations\__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.22). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


loading annotations into memory...
Done (t=0.31s)
creating index...
index created!
Тренировочных филаментов (кропов): 6579


In [6]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO

fused_json_path = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train_fused_deduplicated.json"
train_img_dir = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\train\train_images"

coco = COCO(fused_json_path)
df_folds = pd.read_csv("clean_fused_folds.csv")

val_img_ids = df_folds[df_folds['fold'] == 0]['image_id'].tolist()
train_img_ids = df_folds[df_folds['fold'] != 0]['image_id'].tolist()

train_ann_ids = []
for img_id in train_img_ids:
    train_ann_ids.extend(coco.getAnnIds(imgIds=[img_id]))

print(f"Тренировочных филаментов (кропов): {len(train_ann_ids)}")

# --- 2. Класс Dataset БЕЗ КЭША (чтобы не убить память) ---
class GTCropDataset(Dataset):
    def __init__(self, coco_api, img_dir, ann_ids, transform, pad_ratio=0.20, crop_size=512):
        self.coco = coco_api
        self.img_dir = img_dir
        self.ann_ids = ann_ids
        self.transform = transform
        self.pad_ratio = pad_ratio
        self.crop_size = crop_size

    def __len__(self):
        return len(self.ann_ids)

    def __getitem__(self, idx):
        ann_id = self.ann_ids[idx]
        ann = self.coco.loadAnns([ann_id])[0]
        img_info = self.coco.loadImgs([ann['image_id']])[0]
        fname = img_info['file_name']
        
        # Читаем картинку на лету, БЕЗ сохранения в self.img_cache
        img_path = os.path.join(self.img_dir, fname)
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        disk_mask = raw_img > 15
        proc_img = raw_img.copy()
        if np.any(disk_mask):
            proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
        
        proc_img = proc_img.astype(np.float32) / 255.0
        img = np.stack([proc_img] * 3, axis=-1)
            
        h, w, _ = img.shape
        full_mask = self.coco.annToMask(ann)

        x, y, bw, bh = ann['bbox']
        
        cx, cy = x + bw / 2, y + bh / 2
        max_dim = max(bw, bh) * (1 + self.pad_ratio)
        x1, y1 = max(0, int(cx - max_dim / 2)), max(0, int(cy - max_dim / 2))
        x2, y2 = min(w, int(cx + max_dim / 2)), min(h, int(cy + max_dim / 2))

        crop_img = img[y1:y2, x1:x2]
        crop_mask = full_mask[y1:y2, x1:x2]

        c = cv2.resize(crop_img, (self.crop_size, self.crop_size))
        mk = cv2.resize(crop_mask, (self.crop_size, self.crop_size), interpolation=cv2.INTER_NEAREST).astype('float32')

        out = self.transform(image=c, mask=mk)
        return out['image'], out['mask'].unsqueeze(0).float()

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    ToTensorV2()
])

train_crops = GTCropDataset(coco, train_img_dir, train_ann_ids, train_tf)

# КРИТИЧНО: num_workers=0 предотвращает зависание потоков OpenCV
train_loader = DataLoader(train_crops, batch_size=8, shuffle=True, num_workers=0)

loading annotations into memory...
Done (t=0.46s)
creating index...
index created!
Тренировочных филаментов (кропов): 6579


In [7]:
import segmentation_models_pytorch as smp
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
DEVICE

device(type='cuda')

In [9]:
# Модель (твой U-Net или U-Net++)
seg_model = smp.Unet(encoder_name='resnet34', in_channels=3, classes=1).to(DEVICE)

class TverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha, self.beta, self.smooth = alpha, beta, smooth

    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1)
        t = targets.view(-1)
        tp = (p * t).sum(); fp = (p * (1 - t)).sum(); fn = ((1 - p) * t).sum()
        return 1 - (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)

class BCETverskyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.tv = TverskyLoss()

    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.tv(logits, targets)

criterion = BCETverskyLoss()
optimizer = torch.optim.AdamW(seg_model.parameters(), lr=3e-4)

print("Обучаем U-Net на кропах филаментов...")
seg_model.train()
EPOCHS = 5 

Обучаем U-Net на кропах филаментов...


In [11]:
from tqdm import tqdm
for epoch in range(EPOCHS):
    total_loss = 0
    
    # Оборачиваем train_loader в tqdm для визуализации прогресса
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}")
    
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        
        # Используем Mixed Precision (fp16) для ускорения инференса и экономии памяти GPU
        with torch.amp.autocast('cuda'):
            logits = seg_model(x)
            loss = criterion(logits, y)
            
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Обновляем Loss в прогресс-баре
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    print(f"Средний Loss за эпоху {epoch+1}: {total_loss/len(train_loader):.4f}")

Эпоха 1/5:   0%|          | 0/823 [00:00<?, ?it/s]

Эпоха 1/5: 100%|██████████| 823/823 [08:55<00:00,  1.54it/s, loss=0.2069]


Средний Loss за эпоху 1: 0.2700


Эпоха 2/5: 100%|██████████| 823/823 [08:50<00:00,  1.55it/s, loss=0.2067]


Средний Loss за эпоху 2: 0.2303


...

Средний Loss за эпоху 3: 0.2164


Эпоха 4/5: 100%|██████████| 823/823 [08:55<00:00,  1.54it/s, loss=0.2129]


Средний Loss за эпоху 4: 0.2387


Эпоха 5/5: 100%|██████████| 823/823 [08:50<00:00,  1.55it/s, loss=0.2128]

Средний Loss за эпоху 5: 0.2256


In [12]:
def iou_matrix(gt_masks, pred_masks):
    if not gt_masks or not pred_masks:
        return np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float32)
    G = torch.from_numpy(np.stack([m.ravel() for m in gt_masks])).to(DEVICE).half()
    P = torch.from_numpy(np.stack([m.ravel() for m in pred_masks])).to(DEVICE).half()
    inter = (G @ P.t()).float()
    union = G.sum(1).float()[:, None] + P.sum(1).float()[None, :] - inter
    out = torch.where(union > 0, inter / union, torch.zeros_like(inter)).cpu().numpy()
    del G, P
    return out


class PQAccumulator:
    def __init__(self, thr=0.5):
        self.thr, self.tp_ious, self.fp, self.fn = thr, [], 0, 0

    def add(self, iou):
        n_gt, n_pred = iou.shape
        if n_gt == 0:
            self.fp += n_pred; return
        if n_pred == 0:
            self.fn += n_gt; return
        hit = iou > self.thr
        self.tp_ious.extend(iou[hit].tolist())
        self.fp += int((hit.sum(0) == 0).sum())
        self.fn += int((hit.sum(1) == 0).sum())

    def result(self):
        tp = len(self.tp_ious)
        den = tp + 0.5 * self.fp + 0.5 * self.fn
        sq = sum(self.tp_ious) / tp if tp else 0.0
        dq = tp / den if den else 0.0
        return dict(PQ=dq * sq, DQ=dq, SQ=sq, TP=tp, FP=self.fp, FN=self.fn)


def gt_masks_by_annotator(fname):
    # {image_id: [máscaras]} — un juego por anotador de esta foto
    return {iid: [coco.annToMask(a).astype(bool)
                  for a in coco.loadAnns(coco.getAnnIds(imgIds=[iid]))]
            for iid in photo_to_ids[fname]}

In [13]:
import time

@torch.no_grad()
def run_oracle_experiment(model, val_img_ids, coco_api, img_dir, device, conf_thr=0.35, pad_ratio=0.20):
    model.eval()
    acc = PQAccumulator(thr=0.5)
    t0 = time.time()
    
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    print(f"\nЗапуск Oracle Experiment на {len(val_img_ids)} валидационных изображениях...")
    
    for n, img_id in enumerate(val_img_ids, 1):
        img_info = coco_api.loadImgs([img_id])[0]
        fname = img_info['file_name']
        img_path = os.path.join(img_dir, fname)
        
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        h, w = raw_img.shape
        
        # Fused обработка
        disk_mask = raw_img > 15
        proc_img = raw_img.copy()
        if np.any(disk_mask):
            proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
        proc_img = proc_img.astype(np.float32) / 255.0
        fused_img = np.stack([proc_img] * 3, axis=-1)
        
        # GT аннотации
        ann_ids = coco_api.getAnnIds(imgIds=[img_id])
        gt_anns = coco_api.loadAnns(ann_ids)
        
        # GT Маски для матрицы IoU
        gt_masks = [coco_api.annToMask(a).astype(bool) for a in gt_anns]
        pred_masks = []
        
        # Проходим по каждому GT-боксу (эмулируем идеальный детектор)
        for ann in gt_anns:
            x, y, bw, bh = ann['bbox']
            pw, ph = bw * pad_ratio, bh * pad_ratio
            cx1, cy1 = max(0, int(x - pw)), max(0, int(y - ph))
            cx2, cy2 = min(w, int(x + bw + pw)), min(h, int(y + bh + ph))
            
            if cx2 <= cx1 + 1 or cy2 <= cy1 + 1:
                continue
                
            crop = cv2.resize(fused_img[cy1:cy2, cx1:cx2], (512, 512))
            crop_tensor = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).float().to(device)
            
            logits = model(crop_tensor)
            probs = torch.sigmoid(logits).float().cpu().numpy()[0, 0]
            
            # Возврат маски на оригинальное разрешение
            probs_resized = cv2.resize(probs, (cx2 - cx1, cy2 - cy1))
            
            m = np.zeros((h, w), dtype=bool)
            m[cy1:cy2, cx1:cx2] = probs_resized > conf_thr
            
            # Отсекаем пересечения (Panoptic strict)
            occupied = np.any(pred_masks, axis=0) if pred_masks else np.zeros((h, w), dtype=bool)
            m &= ~occupied
            
            if m.sum() > 0:
                pred_masks.append(m)
        
        # Подсчет PQ
        acc.add(iou_matrix(gt_masks, pred_masks))
            
        if n % 10 == 0:
            print(f'  {n}/{len(val_img_ids)} обработано · {time.time()-t0:.0f}s')

    result = acc.result()
    print("\n" + "="*50)
    print("ORACLE EXPERIMENT RESULTS (GT Boxes -> U-Net)")
    print("="*50)
    print(f"PQ: {result['PQ']:.4f} | SQ: {result['SQ']:.4f} | DQ: {result['DQ']:.4f}")
    print(f"TP: {result['TP']} | FP: {result['FP']} | FN: {result['FN']}")
    print("="*50)

# Запускаем!
run_oracle_experiment(seg_model, val_img_ids, coco, train_img_dir, DEVICE)


Запуск Oracle Experiment на 142 валидационных изображениях...
  10/142 обработано · 5s
  20/142 обработано · 13s
  30/142 обработано · 19s
  40/142 обработано · 25s
  50/142 обработано · 30s
  60/142 обработано · 36s
  70/142 обработано · 44s
  80/142 обработано · 49s
  90/142 обработано · 52s
  100/142 обработано · 56s
  110/142 обработано · 62s
  120/142 обработано · 66s
  130/142 обработано · 71s
  140/142 обработано · 76s

ORACLE EXPERIMENT RESULTS (GT Boxes -> U-Net)
PQ: 0.2723 | SQ: 0.6131 | DQ: 0.4442
TP: 742 | FP: 979 | FN: 878


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

EPOCHS = 12
optimizer = optim.AdamW(seg_model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Запуск полного обучения U-Net на {EPOCHS} эпох...")
seg_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS}")
    
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            logits = seg_model(x)
            loss = criterion(logits, y)
            
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    scheduler.step() 
    print(f"Средний Loss: {total_loss/len(train_loader):.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Сохраняем веса сегментатора
torch.save(seg_model.state_dict(), '/best_seg.pth')
print("Обучение завершено, веса сохранены!")

Запуск полного обучения U-Net на 12 эпох...


Эпоха 1/12: 100%|██████████| 823/823 [08:47<00:00,  1.56it/s, loss=0.2226]


Средний Loss: 0.2254 | LR: 0.000295


Эпоха 2/12: 100%|██████████| 823/823 [08:42<00:00,  1.57it/s, loss=0.1961]


Средний Loss: 0.2149 | LR: 0.000280


...

Средний Loss: 0.2592 | LR: 0.000020


Эпоха 11/12: 100%|██████████| 823/823 [08:56<00:00,  1.53it/s, loss=0.2927]


Средний Loss: 0.2951 | LR: 0.000005


Эпоха 12/12: 100%|██████████| 823/823 [08:57<00:00,  1.53it/s, loss=0.2699]


Средний Loss: 0.2522 | LR: 0.000000
Обучение завершено, веса сохранены!


In [17]:
from ultralytics import YOLO

# Загружаем лучшую модель детектора (проверьте путь, если он отличается)
YOLO_WEIGHTS = r"X:\Desktop\kaggle\solar_project_cv\models\best_seg.pth"
det_model = YOLO(YOLO_WEIGHTS)
seg_model.eval()

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [19]:
class CFG:
    SEED = 42

    # ---------- Etapa 1: detector ----------
    DETECTOR      = 'yolo11m.pt'   # 'yolo11s' más rápido · 'yolo11l/x' necesita BATCH=2-4 o revienta en T4
    DET_IMGSZ     = 1280
    DET_BATCH     = 8              # 16 con un modelo 'x' a 1280 NO entra en 16 GB
    DET_EPOCHS    = 40
    DET_PATIENCE  = 8
    DET_LR        = 0.00087

    # ---------- Etapa 2: segmentador de recortes ----------
    ENCODER     = 'efficientnet-b4'
    CROP_SIZE   = 512
    PAD_RATIO   = 0.20
    BOX_JITTER  = 0.00             # ← 0 en V1 a propósito. Subir a 0.12 es el envío 3.
    SEG_EPOCHS  = 12
    SEG_BATCH   = 8
    SEG_PATIENCE = 7
    SEG_LR      = 2e-4
    TVERSKY_ALPHA, TVERSKY_BETA = 0.3, 0.7

    # ---------- Inferencia ----------
    INFER_IMGSZ = 1024             # ← DEFECTO DELIBERADO (ver cabecera). Envío 2: igualar a DET_IMGSZ.
    CONF_MIN    = 0.05             # piso para cachear; el barrido filtra por encima
    MAX_DET     = 60
    MIN_AREA    = 0                # 0 = desactivado en V1. Activarlo es el envío 4.
    CONF_GRID   = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
    MASK_GRID   = [0.40, 0.50, 0.60]
    SWEEP_PHOTOS = 40              # subconjunto de val para el barrido (el PQ final usa val completo)

    VAL_FRACTION = 0.15

RUN_NAME = 'V1_baseline'
print('Config cargada:', RUN_NAME)

Config cargada: V1_baseline


In [22]:
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

@torch.no_grad()
def cache_predictions_fused(image_path, det_model, seg_model, device, pad_ratio=0.20):
    """Прогоняет YOLO и U-Net, возвращает сырые вероятности для каждого бокса"""
    raw_img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    
    if raw_img is None:
        raise ValueError(f"OpenCV не смог прочитать файл: {image_path}. Проверь путь!")
        
    # Бронебойная защита от лишних каналов (если OpenCV вернул 3D массив)
    if len(raw_img.shape) > 2:
        raw_img = raw_img[:, :, 0]
        
    h, w = raw_img.shape[:2] # Берем только высоту и ширину
    
    # Твоя fused-предобработка
    disk_mask = raw_img > 15
    proc_img = raw_img.copy()
    if np.any(disk_mask):
        proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
    proc_img = proc_img.astype(np.float32) / 255.0
    fused_img = np.stack([proc_img] * 3, axis=-1)

    # 1. YOLO ищет боксы
    r = det_model.predict(str(image_path), imgsz=CFG.DET_IMGSZ, conf=0.05, verbose=False)[0]
    boxes = r.boxes.xyxy.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()

    if len(boxes) == 0:
        return [], (h, w)

    crops, metas = [], []
    for (x1, y1, x2, y2), c in zip(boxes, confs):
        bw, bh = x2 - x1, y2 - y1
        pw, ph = bw * pad_ratio, bh * pad_ratio
        cx1, cy1 = max(0, int(x1 - pw)), max(0, int(y1 - ph))
        cx2, cy2 = min(w, int(x2 + pw)), min(h, int(y2 + ph))
        
        if cx2 <= cx1 + 1 or cy2 <= cy1 + 1:
            continue
            
        crop = cv2.resize(fused_img[cy1:cy2, cx1:cx2], (CFG.CROP_SIZE, CFG.CROP_SIZE))
        crops.append(crop)
        metas.append((float(c), (cx1, cy1, cx2, cy2)))

    out = []
    # 2. U-Net сегментирует кропы батчами для скорости
    for i in range(0, len(crops), 8):
        batch_crops = crops[i:i+8]
        batch_tensor = torch.from_numpy(np.stack(batch_crops)).permute(0, 3, 1, 2).float().to(device)
        
        with torch.amp.autocast('cuda'):
            probs = torch.sigmoid(seg_model(batch_tensor)).float().cpu().numpy()[:, 0]
            
        for p, (c, box) in zip(probs, metas[i:i+8]):
            cx1, cy1, cx2, cy2 = box
            probs_resized = cv2.resize(p, (cx2 - cx1, cy2 - cy1)).astype('float16')
            out.append({'conf': c, 'box': box, 'probs': probs_resized})
            
    return out, (h, w)

def paint_panoptic(cached, shape, conf_t, mask_t, min_area=30):
    """Собирает непересекающиеся маски на холст с учетом порогов"""
    h, w = shape
    # Сортировка по уверенности YOLO: самые надежные боксы рисуются первыми
    items = sorted([it for it in cached if it['conf'] >= conf_t], key=lambda d: -d['conf'])
    occupied = np.zeros((h, w), dtype=bool)
    masks = []
    
    for it in items:
        cx1, cy1, cx2, cy2 = it['box']
        m = np.zeros((h, w), dtype=bool)
        m[cy1:cy2, cx1:cx2] = it['probs'].astype('float32') > mask_t
        m &= ~occupied # Убираем пересечения
        
        if m.sum() <= min_area:
            continue
            
        occupied |= m
        masks.append(m)
        
    return masks

In [24]:
import os
import yaml
import shutil
import pandas as pd
from pycocotools.coco import COCO
from ultralytics import YOLO

# === 1. УКАЖИ СВОИ ПУТИ ЗДЕСЬ ===
BASE_DIR = r"X:\Desktop\kaggle\solar_project_cv"
IMG_DIR = os.path.join(BASE_DIR, r"notebooks\MAGFiLO_1.0_Kaggle_2026\train\train_images")
JSON_PATH = os.path.join(BASE_DIR, r"notebooks\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train_fused_deduplicated.json")
FOLDS_CSV = "clean_fused_folds.csv" # Файл с фолдами, который мы создали ранее

YOLO_DATA_DIR = os.path.join(BASE_DIR, "yolo_dataset")
# ================================

print("1. Создаем структуру папок для YOLO...")
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATA_DIR, f'images/{split}'), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, f'labels/{split}'), exist_ok=True)

print("2. Загружаем разметку и генерируем YOLO-формат...")
coco = COCO(JSON_PATH)
df_folds = pd.read_csv(FOLDS_CSV)

# Fold 0 пойдет в валидацию, остальные в обучение
for _, row in df_folds.iterrows():
    img_id = row['image_id']
    fname = row['file_name']
    split = 'val' if row['fold'] == 0 else 'train'
    
    # Копируем картинку
    src_img = os.path.join(IMG_DIR, fname)
    dst_img = os.path.join(YOLO_DATA_DIR, f'images/{split}', fname)
    if not os.path.exists(dst_img):
        shutil.copy(src_img, dst_img)
        
    # Генерируем .txt лейблы для YOLO
    img_info = coco.loadImgs([img_id])[0]
    w_img, h_img = img_info['width'], img_info['height']
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    
    txt_path = os.path.join(YOLO_DATA_DIR, f'labels/{split}', fname.replace('.jpeg', '.txt').replace('.jpg', '.txt'))
    with open(txt_path, 'w') as f:
        for ann in anns:
            x, y, bw, bh = ann['bbox']
            # Перевод COCO (x, y, w, h) -> YOLO (x_center, y_center, w, h) нормализованные от 0 до 1
            xc = (x + bw / 2) / w_img
            yc = (y + bh / 2) / h_img
            wn = bw / w_img
            hn = bh / h_img
            f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

print("3. Авто-генерация файла конфигурации (data.yaml)...")
yaml_path = os.path.join(YOLO_DATA_DIR, 'data.yaml')
yaml_content = {
    'path': YOLO_DATA_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'filament'}
}
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

1. Создаем структуру папок для YOLO...
2. Загружаем разметку и генерируем YOLO-формат...
loading annotations into memory...
Done (t=0.68s)
creating index...
index created!
3. Авто-генерация файла конфигурации (data.yaml)...


In [25]:
print("4. Запуск обучения детектора YOLO11m!")
det = YOLO('yolo11m.pt') # Скачивает веса YOLO11

# Обучаем (на локальном ПК 20-30 эпох должно хватить)
res = det.train(
    data=yaml_path,
    epochs=30,
    imgsz=1280,
    batch=4, 
    patience=8,
    project=os.path.join(BASE_DIR, "models"),
    name='yolo_detector'
)

4. Запуск обучения детектора YOLO11m!
New https://pypi.org/project/ultralytics/8.4.155 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=X:\Desktop\kaggle\solar_project_cv\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt

c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1357: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1359: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(Tr

AMP: checks passed 
train: Fast image access  (ping: 0.10.0 ms, read: 81.418.6 MB/s, size: 771.9 KB)
train: Scanning X:\Desktop\kaggle\solar_project_cv\yolo_dataset\labels\train... 565 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 565/565 513.3it/s 1.1s0.1s
train: New cache created: X:\Desktop\kaggle\solar_project_cv\yolo_dataset\labels\train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 97.122.7 MB/s, size: 716.5 KB)
val: Scanning X:\Desktop\kaggle\solar_project_cv\yolo_dataset\labels\val... 142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 142/142 288.8it/s 0.5s0.1s
val: New cache created: X:\Desktop\kaggle\solar_project_cv\yolo_dataset\labels\val.cache
Plotting labels to X:\Desktop\kaggle\solar_project_cv\models\yolo_detector\labels.jpg... 
optimizer: '

c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1357: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1359: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(Tr

       1/30      8.57G      1.807      2.493      1.426          2       1280: 100% ━━━━━━━━━━━━ 142/142 0.5it/s 4:381.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.2it/s 8.2s0.4s
                   all        142       1620      0.125      0.311     0.0757     0.0326

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      29/30      3.05G      1.238      1.132      1.177         15       1280: 100% ━━━━━━━━━━━━ 142/142 0.7it/s 3:221.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.0it/s 9.1s0.4s
                   all        142       1620      0.608      0.528      0.546      0.322

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1357: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\ultralytics\nn\modules\block.py:1359: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(Tr

      30/30      6.44G      1.038     0.9479       1.06         42       1280: 0% ──────────── 0/142  2.0s

c:\Users\User\anaconda3\envs\tensorflowgpu\lib\site-packages\torch\autograd\graph.py:825: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:208.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      8.33G      1.233      1.106      1.174         23       1280: 100% ━━━━━━━━━━━━ 142/142 0.6it/s 4:111.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.0it/s 9.0s0.4s
                   all        142       1620      0.609       0.52      0.542      0.322

30 epochs completed in 2.019 hours.
Optimizer stripped from X:\Desktop\kaggle\solar_project_cv\models\yolo_detector\weights\last.pt, 40.7MB
Optimizer stripped from X:\Desktop\kaggle\solar_project_cv\models\yolo_detector\weights\best.pt, 40.7MB

Validating X:\Desktop\kaggle\solar_project_cv\models\yolo_detector\weights\best.pt...
Ultralytics 8.3.217  Python-3.10.14 torch-2.5.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.6it/s 6.9s0.4s
  

In [26]:
print(f"Готово! Веса детектора сохранены в: {res.save_dir}/weights/best.pt")

Готово! Веса детектора сохранены в: X:\Desktop\kaggle\solar_project_cv\models\yolo_detector/weights/best.pt


In [ ]:
# import os
# import cv2
# import time
# import torch
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# from ultralytics import YOLO
# import segmentation_models_pytorch as smp

In [27]:
# === 1. ПУТИ И НАСТРОЙКИ ===
BASE_DIR = r"X:\Desktop\kaggle\solar_project_cv"
IMG_DIR = os.path.join(BASE_DIR, r"notebooks\MAGFiLO_1.0_Kaggle_2026\train\train_images")
FOLDS_CSV = "clean_fused_folds.csv"

# ВНИМАНИЕ: Проверь, правильный ли путь к весам U-Net
YOLO_WEIGHTS = os.path.join(BASE_DIR, r"models\yolo_detector\weights\best.pt")
SEG_WEIGHTS = r"X:\Desktop\kaggle\solar_project_cv\models\best_seg.pth" # Измени, если перенес файл!

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG_CROP_SIZE = 512
CFG_DET_IMGSZ = 1280

# === 2. ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ ===
print("Загрузка моделей...")
det_model = YOLO(YOLO_WEIGHTS)

seg_model = smp.Unet(encoder_name='resnet34', in_channels=3, classes=1).to(DEVICE)
seg_model.load_state_dict(torch.load(SEG_WEIGHTS, map_location=DEVICE))
seg_model.eval()

clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

Загрузка моделей...


In [28]:
# === 3. ФУНКЦИИ ИНФЕРЕНСА И МЕТРИК ===
@torch.no_grad()
def cache_predictions_fused(image_path, det_model, seg_model, device, pad_ratio=0.20):
    raw_img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if len(raw_img.shape) > 2: raw_img = raw_img[:, :, 0]
    h, w = raw_img.shape[:2]
    
    disk_mask = raw_img > 15
    proc_img = raw_img.copy()
    if np.any(disk_mask):
        proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
    fused_img = np.stack([proc_img.astype(np.float32) / 255.0] * 3, axis=-1)

    r = det_model.predict(str(image_path), imgsz=CFG_DET_IMGSZ, conf=0.05, verbose=False)[0]
    boxes, confs = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()

    if len(boxes) == 0: return [], (h, w)

    crops, metas = [], []
    for (x1, y1, x2, y2), c in zip(boxes, confs):
        bw, bh = x2 - x1, y2 - y1
        pw, ph = bw * pad_ratio, bh * pad_ratio
        cx1, cy1 = max(0, int(x1 - pw)), max(0, int(y1 - ph))
        cx2, cy2 = min(w, int(x2 + pw)), min(h, int(y2 + ph))
        if cx2 <= cx1 + 1 or cy2 <= cy1 + 1: continue
            
        crop = cv2.resize(fused_img[cy1:cy2, cx1:cx2], (CFG_CROP_SIZE, CFG_CROP_SIZE))
        crops.append(crop)
        metas.append((float(c), (cx1, cy1, cx2, cy2)))

    out = []
    for i in range(0, len(crops), 8):
        batch = torch.from_numpy(np.stack(crops[i:i+8])).permute(0, 3, 1, 2).float().to(device)
        with torch.amp.autocast('cuda'):
            probs = torch.sigmoid(seg_model(batch)).float().cpu().numpy()[:, 0]
        for p, (c, box) in zip(probs, metas[i:i+8]):
            cx1, cy1, cx2, cy2 = box
            probs_resized = cv2.resize(p, (cx2 - cx1, cy2 - cy1)).astype('float16')
            out.append({'conf': c, 'box': box, 'probs': probs_resized})
    return out, (h, w)

def paint_panoptic(cached, shape, conf_t, mask_t, min_area=30):
    h, w = shape
    items = sorted([it for it in cached if it['conf'] >= conf_t], key=lambda d: -d['conf'])
    occupied = np.zeros((h, w), dtype=bool)
    masks = []
    for it in items:
        cx1, cy1, cx2, cy2 = it['box']
        m = np.zeros((h, w), dtype=bool)
        m[cy1:cy2, cx1:cx2] = it['probs'].astype('float32') > mask_t
        m &= ~occupied
        if m.sum() > min_area:
            occupied |= m
            masks.append(m)
    return masks

def iou_matrix(gt_masks, pred_masks):
    if not gt_masks or not pred_masks:
        return np.zeros((len(gt_masks), len(pred_masks)), dtype=np.float32)
    G = torch.from_numpy(np.stack([m.ravel() for m in gt_masks])).to(DEVICE).half()
    P = torch.from_numpy(np.stack([m.ravel() for m in pred_masks])).to(DEVICE).half()
    inter = (G @ P.t()).float()
    union = G.sum(1).float()[:, None] + P.sum(1).float()[None, :] - inter
    out = torch.where(union > 0, inter / union, torch.zeros_like(inter)).cpu().numpy()
    del G, P; return out

class PQAccumulator:
    def __init__(self, thr=0.5):
        self.thr, self.tp_ious, self.fp, self.fn = thr, [], 0, 0
    def add(self, iou):
        n_gt, n_pred = iou.shape
        if n_gt == 0: self.fp += n_pred; return
        if n_pred == 0: self.fn += n_gt; return
        hit = iou > self.thr
        self.tp_ious.extend(iou[hit].tolist())
        self.fp += int((hit.sum(0) == 0).sum())
        self.fn += int((hit.sum(1) == 0).sum())
    def result(self):
        tp = len(self.tp_ious)
        den = tp + 0.5 * self.fp + 0.5 * self.fn
        sq = sum(self.tp_ious) / tp if tp else 0.0
        dq = tp / den if den else 0.0
        return dict(PQ=dq * sq, DQ=dq, SQ=sq, TP=tp, FP=self.fp, FN=self.fn)

def gt_masks_by_annotator(fname, coco_api, photo_to_ids):
    return {iid: [coco_api.annToMask(a).astype(bool) for a in coco_api.loadAnns(coco_api.getAnnIds(imgIds=[iid]))]
            for iid in photo_to_ids.get(fname, [])}


In [29]:
# === 4. ЗАПУСК GRID SEARCH ===
df_folds = pd.read_csv(FOLDS_CSV)
val_photos = df_folds[df_folds['fold'] == 0]['file_name'].unique()[:40] # Берем 40 картинок для скорости

from pycocotools.coco import COCO
JSON_PATH = os.path.join(BASE_DIR, r"notebooks\MAGFiLO_1.0_Kaggle_2026\train\MAGFiLO_1.0_Annotations_kaggle2026_train_fused_deduplicated.json")
coco = COCO(JSON_PATH)
photo_to_ids = {}
for iid in coco.imgs:
    photo_to_ids.setdefault(coco.imgs[iid]['file_name'], []).append(iid)

conf_grid = [0.15, 0.20, 0.25, 0.30]
mask_grid = [0.40, 0.50, 0.60]
grid = [(c, m) for c in conf_grid for m in mask_grid]
accs = {k: PQAccumulator(thr=0.5) for k in grid}

print(f"\nТестируем {len(grid)} комбинаций на {len(val_photos)} изображениях...")
t0 = time.time()

for n, fname in enumerate(val_photos, 1):
    img_path = os.path.join(IMG_DIR, fname)
    cached, shape = cache_predictions_fused(img_path, det_model, seg_model, DEVICE)
    gts = gt_masks_by_annotator(fname, coco, photo_to_ids)
    
    for combo in grid:
        preds = paint_panoptic(cached, shape, *combo)
        for masks in gts.values():
            accs[combo].add(iou_matrix(masks, preds))
            
    if n % 10 == 0:
        print(f"  {n}/{len(val_photos)} обработано · {time.time()-t0:.0f}s")

loading annotations into memory...
Done (t=0.39s)
creating index...
index created!

Тестируем 12 комбинаций на 40 изображениях...
  10/40 обработано · 24s
  20/40 обработано · 62s
  30/40 обработано · 94s
  40/40 обработано · 122s


In [30]:
# Вывод результатов
results = []
for c, m in grid:
    res = accs[(c, m)].result()
    results.append({'conf': c, 'mask_thr': m, **res})

sweep = pd.DataFrame(results).sort_values('PQ', ascending=False).reset_index(drop=True)
print("\n=== ЛУЧШИЕ РЕЗУЛЬТАТЫ (YOLO + U-Net) ===")
display(sweep.head(5))


=== ЛУЧШИЕ РЕЗУЛЬТАТЫ (YOLO + U-Net) ===


,conf,mask_thr,PQ,DQ,SQ,TP,FP,FN
0,0.30,0.6,0.417952,0.635965,0.657193,290,133,199
1,0.30,0.5,0.415562,0.632159,0.657369,287,132,202
2,0.25,0.6,0.410105,0.625393,0.655756,298,166,191
3,0.25,0.5,0.408931,0.623815,0.655532,296,164,193
4,0.30,0.4,0.408393,0.621830,0.656759,282,136,207


In [36]:
import pycocotools.mask as maskUtils
# --- 1. ПУТИ ДЛЯ KAGGLE ---
TEST_IMG_DIR = r"X:\Desktop\kaggle\solar_project_cv\notebooks\MAGFiLO_1.0_Kaggle_2026\test\test_images"
YOLO_WEIGHTS = r"X:\Desktop\kaggle\solar_project_cv\models\yolo_detector\weights\best.pt"
SEG_WEIGHTS = r"X:\Desktop\kaggle\solar_project_cv\models\best_seg.pth"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BEST_CONF = 0.30
BEST_MASK_THR = 0.60
CFG_CROP_SIZE = 512
CFG_DET_IMGSZ = 1280
PAD_RATIO = 0.20

# --- 2. ИНИЦИАЛИЗАЦИЯ ---
det_model = YOLO(YOLO_WEIGHTS)
seg_model = smp.Unet(encoder_name='resnet34', in_channels=3, classes=1).to(DEVICE)
seg_model.load_state_dict(torch.load(SEG_WEIGHTS, map_location=DEVICE))
seg_model.eval()
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

# НОВАЯ ФУНКЦИЯ КОДИРОВАНИЯ (COCO Compressed RLE)
def coco_rle_encode(mask):
    # pycocotools требует массив Fortran-порядка и тип uint8
    rle = maskUtils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle['counts'].decode('utf-8')

# --- 3. ИНФЕРЕНС И ФОРМИРОВАНИЕ САБМИТА ---
test_images = sorted(os.listdir(TEST_IMG_DIR))
submission_data = []

for fname in test_images:
    img_path = os.path.join(TEST_IMG_DIR, fname)
    raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if len(raw_img.shape) > 2: raw_img = raw_img[:, :, 0]
    h, w = raw_img.shape[:2]
    
    # Fused preprocessing
    disk_mask = raw_img > 15
    proc_img = raw_img.copy()
    if np.any(disk_mask):
        proc_img[disk_mask] = clahe.apply(raw_img)[disk_mask]
    fused_img = np.stack([proc_img.astype(np.float32) / 255.0] * 3, axis=-1)

    # Детекция YOLO
    r = det_model.predict(img_path, imgsz=CFG_DET_IMGSZ, conf=BEST_CONF, verbose=False)[0]
    boxes, confs = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()

    occupied = np.zeros((h, w), dtype=bool)
    
    filename_base = fname.rsplit('.', 1)[0]
    filament_counter = 1

    sorted_indices = np.argsort(-confs)
    for idx in sorted_indices:
        x1, y1, x2, y2 = boxes[idx]
        bw, bh = x2 - x1, y2 - y1
        pw, ph = bw * PAD_RATIO, bh * PAD_RATIO
        cx1, cy1 = max(0, int(x1 - pw)), max(0, int(y1 - ph))
        cx2, cy2 = min(w, int(x2 + pw)), min(h, int(y2 + ph))
        
        if cx2 <= cx1 + 1 or cy2 <= cy1 + 1: continue
            
        crop = cv2.resize(fused_img[cy1:cy2, cx1:cx2], (CFG_CROP_SIZE, CFG_CROP_SIZE))
        batch = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
        
        with torch.no_grad(), torch.amp.autocast('cuda'):
            prob = torch.sigmoid(seg_model(batch)).float().cpu().numpy()[0, 0]
            
        prob_resized = cv2.resize(prob, (cx2 - cx1, cy2 - cy1))
        
        m = np.zeros((h, w), dtype=bool)
        m[cy1:cy2, cx1:cx2] = prob_resized > BEST_MASK_THR
        m &= ~occupied
        
        if m.sum() > 30:
            occupied |= m
            submission_data.append({
                'filament_id': f"{filename_base}_{filament_counter}", 
                'segmentation_rle': coco_rle_encode(m)
            })
            filament_counter += 1

# Сохранение (Kaggle больше не увидит Null значений)
df_sub = pd.DataFrame(submission_data)

In [37]:
df_sub.to_csv('submission5.csv', index=False)
print("submission.csv успешно создан в правильном формате!")

submission.csv успешно создан в правильном формате!
